# UW-LYT-MS V2 (Universal: Kaggle & Local)
Runner for the RGB-only, LYT-inspired multiscale UW-LYT V2 model (~125k parameters).
This notebook automatically detects whether it is running on **Kaggle** or **Local machine** (Windows/Linux/macOS), and handles GPU/CPU devices smoothly.

In [ ]:
import sys
try:
    import thop
except ImportError:
    !{sys.executable} -m pip install thop -q

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Check if running on Kaggle or Local
IS_KAGGLE = Path("/kaggle").exists()

if IS_KAGGLE:
    REPO = Path("/kaggle/working/underwater-image-enhancement")
    if not REPO.exists():
        subprocess.run([
            "git", "clone", "--branch", "refactored-paper-core",
            "--single-branch", "--depth", "1",
            "https://github.com/heniath/underwater-image-enhancement.git", str(REPO),
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    os.chdir(REPO)
    !{sys.executable} -m pip install -q -e .
else:
    # Running on local machine
    REPO = Path.cwd()
    if not (REPO / "src" / "uwir").exists() and (REPO.parent / "src" / "uwir").exists():
        REPO = REPO.parent
    os.chdir(REPO)
    if str(REPO / "src") not in sys.path:
        sys.path.insert(0, str(REPO / "src"))

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Active Repo : {REPO.resolve()}")
print(f"Python Exec : {sys.executable}")

# Verify UW-LYT-MS V2 is available
from uwir.models.registry import ALL_MODEL_NAMES
assert "uwlytmsv2_3ch" in ALL_MODEL_NAMES, "uwlytmsv2_3ch not found in registry!"
print("[OK] uwlytmsv2_3ch registered and ready.")

In [ ]:
import os
import torch
from pathlib import Path

candidates = [
    Path("/kaggle/input/euvp-dataset/EUVP"),
    Path("/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP"),
    REPO / "datasets/EUVP",
    REPO.parent / "datasets/EUVP",
    Path("D:/datasets/EUVP"),
    Path("C:/datasets/EUVP"),
]
EUVP_ROOT = next((p for p in candidates if p.exists() and (p / "test_samples/Inp").exists()), None)
if EUVP_ROOT is None and IS_KAGGLE:
    matches = list(Path("/kaggle/input").glob("**/EUVP/test_samples"))
    EUVP_ROOT = matches[0].parent if matches else None

if EUVP_ROOT is None:
    print("[WARN] Full EUVP dataset not found. Generating minimal dummy smoke dataset for pipeline validation...")
    EUVP_ROOT = REPO / "datasets/EUVP_mini_smoke"
    for split in ["underwater_imagenet", "underwater_dark", "underwater_scenes"]:
        (EUVP_ROOT / "Paired" / split / "trainA").mkdir(parents=True, exist_ok=True)
        (EUVP_ROOT / "Paired" / split / "trainB").mkdir(parents=True, exist_ok=True)
    (EUVP_ROOT / "test_samples/Inp").mkdir(parents=True, exist_ok=True)
    (EUVP_ROOT / "test_samples/GTr").mkdir(parents=True, exist_ok=True)
    from PIL import Image
    import numpy as np
    if not list((EUVP_ROOT / "test_samples/Inp").glob("*.jpg")):
        for i in range(4):
            img = Image.fromarray((np.random.rand(256, 256, 3) * 255).astype(np.uint8))
            img.save(EUVP_ROOT / "Paired/underwater_imagenet/trainA" / f"img_{i}.jpg")
            img.save(EUVP_ROOT / "Paired/underwater_imagenet/trainB" / f"img_{i}.jpg")
            img.save(EUVP_ROOT / "test_samples/Inp" / f"test_{i}.jpg")
            img.save(EUVP_ROOT / "test_samples/GTr" / f"test_{i}.jpg")

NUM_GPUS = torch.cuda.device_count()
HAS_CUDA = NUM_GPUS > 0
DEVICE = "cuda" if HAS_CUDA else "cpu"

SMOKE = True  # Set True for quick 1-epoch test, False for full 50-epoch training
EPOCHS, RUNS, SEEDS = (1, 1, "0") if SMOKE else (50, 3, "0 1 2")
TAG = "smoke" if SMOKE else "full"
BATCH_SIZE = 16 if HAS_CUDA else 4
THREADS = 2 if (HAS_CUDA and os.name != "nt") else 0

base_output = Path("/kaggle/working") if IS_KAGGLE else REPO
CHECKPOINTS = base_output / f"checkpoints_uwlytmsv2_{TAG}"
RESULTS = base_output / f"results_uwlytmsv2_{TAG}"

print(f"Dataset Path : {EUVP_ROOT}")
print(f"Device       : {DEVICE} (GPUs: {NUM_GPUS})")
print(f"Mode         : {TAG} (Epochs: {EPOCHS}, Runs: {RUNS}, BatchSize: {BATCH_SIZE})")
print(f"Checkpoints  : {CHECKPOINTS}")
print(f"Results      : {RESULTS}")

In [ ]:
gpu_args = f"--gpu_mode --num_gpus {max(1, NUM_GPUS)}" if HAS_CUDA else "--no_gpu --num_gpus 1"

!{sys.executable} -m scripts.experiments.ablation_euvp \
    --data_train_euvp "{EUVP_ROOT}" \
    --checkpoint_dir "{CHECKPOINTS}" \
    --val_folder "{RESULTS}" \
    --variants uwlytmsv2_3ch \
    --nEpochs {EPOCHS} --batchSize {BATCH_SIZE} --cropSize 256 \
    --lr 1e-4 --weight_decay 1e-5 \
    --L1_weight 1.0 --perceptual_weight 1.0 --SSIM_weight 0.0 \
    --scheduler_step 30 --scheduler_gamma 0.5 \
    --early_stop_patience 20 --num_runs {RUNS} --seeds {SEEDS} \
    --threads {THREADS} {gpu_args}

In [ ]:
eval_gpu = "True" if HAS_CUDA else "False"

!{sys.executable} -m uwir.cli.evaluate \
    --eval_benchmark euvp --data_train_euvp "{EUVP_ROOT}" \
    --checkpoint_dir "{CHECKPOINTS}" \
    --val_folder "{RESULTS / 'evaluation'}" \
    --batchSize {BATCH_SIZE} --cropSize 256 --threads {THREADS} \
    --gpu_mode {eval_gpu} --native_eval True

!{sys.executable} -m uwir.cli.profile uwlytmsv2 unet_5ch --device {DEVICE} --no-pretrained \
    --img-size 256 --runs 5 --output-dir "{RESULTS / 'profile'}"